In [1]:
import torch 
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
X=torch.rand(2,20)
net(X)

tensor([[ 0.0793, -0.0239,  0.0709,  0.1290, -0.0545,  0.3644, -0.0395, -0.3638,
          0.0947,  0.0540],
        [ 0.0284,  0.0632,  0.0297, -0.0218, -0.1802,  0.3441,  0.0404, -0.2319,
         -0.0116, -0.0086]], grad_fn=<AddmmBackward0>)

In [3]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20,256)
        self.out = nn.Linear(256,10)
        
    def forward(self,X):
        return self.out(F.relu(self.hidden(X)))

In [4]:
net = MLP()
net(X)

tensor([[-0.2110, -0.0608,  0.0442, -0.2040, -0.0969, -0.0844, -0.2638,  0.1035,
          0.1975,  0.2096],
        [-0.3326, -0.1768, -0.0008, -0.2760, -0.0858, -0.0679, -0.2883, -0.0201,
          0.1295,  0.2957]], grad_fn=<AddmmBackward0>)

In [5]:
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for idx,module in enumerate(args):
            self._modules[str(idx)] = module
    
    def forward(self,X):
        for block in self._modules.values():
            X = block(X)
        return X

In [6]:
net = MySequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(X)

tensor([[ 0.0076,  0.1570, -0.1604, -0.0314,  0.1123,  0.1235,  0.0024,  0.0005,
          0.1701, -0.1844],
        [ 0.1328,  0.1794, -0.0785,  0.0035,  0.1095, -0.0295, -0.0104, -0.1089,
          0.2255, -0.0730]], grad_fn=<AddmmBackward0>)

In [7]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

- 固定参数与可学习参数分离
    - self.rand_weight 是固定权重（requires_grad=False），仅作为静态变换，不参与模型训练更新。
    - self.linear 是可学习层，其权重和偏置会在反向传播中被优化。
- 层权重共享
    - 前向传播中两次调用 self.linear(X)，使用的是同一组权重参数，实现了层权重的共享。
- 动态计算图
    - 代码中包含 while 循环，循环次数由输入张量的数值动态决定，这体现了 PyTorch 动态计算图的灵活性（计算图结构可随输入变化）。
- 梯度传播范围
    - 输出的 grad_fn=<SumBackward0>表明梯度仅来自可学习的 linear 层，固定权重 rand_weight 不参与梯度计算。


In [10]:
net = FixedHiddenMLP()
net(X)

tensor(0.3417, grad_fn=<SumBackward0>)

In [11]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(-0.1045, grad_fn=<SumBackward0>)